# Structural Model Tests: Does Adding Oil Jumps Help?

**Two approaches that sidestep the credit-spread-level puzzle:**

1. **Approach 1 (Nested Change-on-Change):** Test the *marginal* contribution of the jump component to tracking CDS movements, with DiD interaction for oil exporters × high OVX.

2. **Approach 3 (Residual-Based):** Strip standard risk factors from ΔCDS, then test whether the model's jump signal explains the residual — especially for oil exporters during stress.

**Prerequisite:** Run notebook `08` first so that `df_res` is loaded.

# # Structural Model Validation: Does Adding Oil Jumps Help?
#
# **Core question:** Oil volatility regimes shape sovereign asset volatility.
# If the sovereign depends on oil (exporter), adding this jump information
# to the structural model should make *changes* in model-implied DD/spreads
# more informative about actual CDS movements.
#
# We test this with two complementary approaches that sidestep the
# credit-spread-level puzzle entirely.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

In [ ]:
# ================================================================
# PANEL PREPARATION
# ================================================================
# Assumes df_res exists from notebook 08.
# Adjust column names below if your panel uses different names.

df = df_res.sort_values(['country_clean', 'date']).copy()

# ---- Weekly changes (first-differences) ----
for col in ['cds_spread', 'spread_m1', 'spread_m3', 'dd_m1', 'dd_m3']:
    df[f'd_{col}'] = df.groupby('country_clean')[col].diff()

# ---- The jump increment: what jumps ADD to the model ----
# This is the key variable. It measures how much the jump model's
# DD or spread differs from the baseline, and how that changes week-to-week.
df['dd_jump_increment']  = df['dd_m3'] - df['dd_m1']       # level: always <= 0
df['d_dd_jump_increment'] = df.groupby('country_clean')['dd_jump_increment'].diff()
df['d_spread_jump_incr']  = df['d_spread_m3'] - df['d_spread_m1']

# ---- OVX regime indicator ----
OVX_P75 = df['OVX'].quantile(0.75)
df['high_ovx'] = (df['OVX'] >= OVX_P75).astype(int)
print(f'OVX 75th percentile threshold: {OVX_P75:.1f}')
print(f'High-OVX weeks: {df["high_ovx"].sum()} / {len(df)} '
      f'({df["high_ovx"].mean()*100:.0f}%)')

# ---- Auto-detect control variables ----
# Common controls in sovereign CDS panels. We check which exist.
CANDIDATE_CONTROLS = [
    'vix', 'VIX', 'd_vix', 'vix_change',
    'us_10y', 'd_us10y', 'us_treasury_10y',
    'msci_world_ret', 'msci_em_ret', 'msci_ret',
    'fx_ret', 'fx_return', 'd_fx',
    'brent_ret', 'oil_ret', 'brent_return',
    'embi_spread', 'd_embi',
]

available_controls = [c for c in CANDIDATE_CONTROLS if c in df.columns]
print(f'\nAvailable control variables: {available_controls}')
print(f'Countries: {sorted(df["country_clean"].unique())}')
print(f'Oil exporters: {sorted(df[df["oil_exporter"]==1]["country_clean"].unique())}')
print(f'Controls:      {sorted(df[df["oil_exporter"]==0]["country_clean"].unique())}')

# ---- Drop rows with missing key variables ----
key_cols = ['d_cds_spread', 'd_dd_m1', 'd_dd_m3', 'd_dd_jump_increment',
            'oil_exporter', 'high_ovx']
df_clean = df.dropna(subset=key_cols).copy()
print(f'\nClean panel: {len(df_clean)} obs, '
      f'{df_clean["country_clean"].nunique()} countries')

# ---
# ## Approach 1: Nested Change-on-Change Regression
#
# **Idea:** Instead of comparing R² from two *separate* regressions (M1 vs M3),
# we run a *single* regression that includes both the baseline DD change
# and the incremental jump contribution, then test whether the jump
# increment matters — and whether it matters *more* for oil exporters.
#
# $$\Delta CDS_{it} = \alpha_i + \beta_1 \Delta DD^{baseline}_{it}
#   + \beta_2 \Delta DD^{jump}_{it} + \gamma \cdot X_{it} + \varepsilon_{it}$$
#
# where $\Delta DD^{jump}_{it} = DD^{M3}_{it} - DD^{M1}_{it}$ is the
# marginal contribution of jumps.
#
# **Key test:** $\beta_2 \neq 0$ means jumps add information.
# Then we interact with `oil_exporter` and `high_ovx` to test the DiD.

In [ ]:
# ================================================================
# APPROACH 1: NESTED CHANGE-ON-CHANGE
# ================================================================

print('=' * 80)
print('  APPROACH 1: NESTED CHANGE-ON-CHANGE REGRESSIONS')
print('  ΔCDS = α_i + β₁·ΔDD_baseline + β₂·ΔDD_jump_increment + controls + ε')
print('=' * 80)

# --- 1A. Simple nested test: does the jump increment add info? ---
print('\n--- 1A. Baseline: does the jump increment add information? ---\n')

# Prepare control string for formulas
ctrl_str = ' + '.join(available_controls) if available_controls else ''

# Model A1: Baseline only
fml_base = 'd_cds_spread ~ d_dd_m1'
if ctrl_str:
    fml_base += ' + ' + ctrl_str
fml_base += ' + C(country_clean)'

# Model A2: Baseline + jump increment
fml_nested = 'd_cds_spread ~ d_dd_m1 + d_dd_jump_increment'
if ctrl_str:
    fml_nested += ' + ' + ctrl_str
fml_nested += ' + C(country_clean)'

reg_base = smf.ols(fml_base, data=df_clean).fit(
    cov_type='cluster', cov_kwds={'groups': df_clean['country_clean']})
reg_nested = smf.ols(fml_nested, data=df_clean).fit(
    cov_type='cluster', cov_kwds={'groups': df_clean['country_clean']})

print('Model A1 (baseline DD only):')
print(f'  R² = {reg_base.rsquared:.4f},  N = {int(reg_base.nobs)}')
b1 = reg_base.params.get('d_dd_m1', np.nan)
t1 = reg_base.tvalues.get('d_dd_m1', np.nan)
p1 = reg_base.pvalues.get('d_dd_m1', np.nan)
print(f'  β(ΔDD_baseline) = {b1:.4f}  [t = {t1:.2f}, p = {p1:.4f}]')

print('\nModel A2 (baseline DD + jump increment):')
print(f'  R² = {reg_nested.rsquared:.4f},  N = {int(reg_nested.nobs)}')
b1n = reg_nested.params.get('d_dd_m1', np.nan)
t1n = reg_nested.tvalues.get('d_dd_m1', np.nan)
p1n = reg_nested.pvalues.get('d_dd_m1', np.nan)
b2  = reg_nested.params.get('d_dd_jump_increment', np.nan)
t2  = reg_nested.tvalues.get('d_dd_jump_increment', np.nan)
p2  = reg_nested.pvalues.get('d_dd_jump_increment', np.nan)
print(f'  β(ΔDD_baseline)  = {b1n:.4f}  [t = {t1n:.2f}, p = {p1n:.4f}]')
print(f'  β(ΔDD_jump_incr) = {b2:.4f}  [t = {t2:.2f}, p = {p2:.4f}]')
print(f'\n  R² improvement: {reg_nested.rsquared - reg_base.rsquared:+.4f}')

# F-test for nested model (is the jump increment jointly significant?)
from statsmodels.stats.anova import anova_lm
try:
    # Re-fit without cluster for proper F-test
    reg_base_ols = smf.ols(fml_base, data=df_clean).fit()
    reg_nested_ols = smf.ols(fml_nested, data=df_clean).fit()
    f_test = anova_lm(reg_base_ols, reg_nested_ols)
    f_stat = f_test['F'].iloc[1]
    f_pval = f_test['Pr(>F)'].iloc[1]
    print(f'  F-test (jump increment = 0): F = {f_stat:.2f}, p = {f_pval:.4f}')
except Exception as e:
    print(f'  (F-test skipped: {e})')

In [ ]:
# --- 1B. DiD: Is the jump increment more useful for oil exporters? ---
print('\n' + '=' * 80)
print('--- 1B. DiD: Jump increment × Oil Exporter × High OVX ---')
print('=' * 80)

# Full interaction model
fml_did = ('d_cds_spread ~ d_dd_m1 '
           '+ d_dd_jump_increment '
           '+ d_dd_jump_increment : oil_exporter '
           '+ d_dd_jump_increment : high_ovx '
           '+ d_dd_jump_increment : oil_exporter : high_ovx')
if ctrl_str:
    fml_did += ' + ' + ctrl_str
fml_did += ' + C(country_clean)'

reg_did = smf.ols(fml_did, data=df_clean).fit(
    cov_type='cluster', cov_kwds={'groups': df_clean['country_clean']})

print(f'\nR² = {reg_did.rsquared:.4f},  N = {int(reg_did.nobs)}\n')

# Extract and display the key coefficients
key_vars = ['d_dd_m1', 'd_dd_jump_increment',
            'd_dd_jump_increment:oil_exporter',
            'd_dd_jump_increment:high_ovx',
            'd_dd_jump_increment:oil_exporter:high_ovx']

print(f'  {"Variable":<50s} {"Coef":>8s} {"t":>7s} {"p":>7s}  {"Sig":>3s}')
print('  ' + '-' * 80)
for var in key_vars:
    if var in reg_did.params.index:
        coef = reg_did.params[var]
        tval = reg_did.tvalues[var]
        pval = reg_did.pvalues[var]
        sig = '***' if pval < 0.01 else '**' if pval < 0.05 else '*' if pval < 0.1 else ''
        print(f'  {var:<50s} {coef:>8.4f} {tval:>7.2f} {pval:>7.4f}  {sig:>3s}')

# --- Interpretation guide ---
print('\n  INTERPRETATION:')
print('  β(ΔDD_jump_incr)               = effect of jumps for controls in low OVX')
print('  β(... × oil_exporter)           = additional effect for oil exporters')
print('  β(... × high_ovx)               = additional effect during high OVX')
print('  β(... × oil_exporter × high_ovx)= DiD: extra effect for oil exp. in high OVX')
print('  → THESIS HYPOTHESIS: last coefficient should be significant')

In [ ]:
# --- 1C. Split-sample: run separately for oil exporters vs controls ---
print('\n' + '=' * 80)
print('--- 1C. Split-Sample Regressions ---')
print('=' * 80)

for grp, label in [(1, 'Oil Exporters'), (0, 'Controls')]:
    sub = df_clean[df_clean['oil_exporter'] == grp]
    if len(sub) < 50:
        print(f'\n  {label}: insufficient data (n={len(sub)})')
        continue

    for regime, rlabel in [('all', 'Full Sample'),
                            ('high', 'High OVX Only'),
                            ('low', 'Low OVX Only')]:
        if regime == 'high':
            ss = sub[sub['high_ovx'] == 1]
        elif regime == 'low':
            ss = sub[sub['high_ovx'] == 0]
        else:
            ss = sub

        if len(ss) < 30:
            continue

        fml = 'd_cds_spread ~ d_dd_m1 + d_dd_jump_increment'
        if ctrl_str:
            fml += ' + ' + ctrl_str
        fml += ' + C(country_clean)'

        try:
            reg = smf.ols(fml, data=ss).fit(
                cov_type='cluster', cov_kwds={'groups': ss['country_clean']})

            b_base = reg.params.get('d_dd_m1', np.nan)
            b_jump = reg.params.get('d_dd_jump_increment', np.nan)
            t_jump = reg.tvalues.get('d_dd_jump_increment', np.nan)
            p_jump = reg.pvalues.get('d_dd_jump_increment', np.nan)
            sig = '***' if p_jump < 0.01 else '**' if p_jump < 0.05 else '*' if p_jump < 0.1 else ''

            print(f'\n  {label:<18s} | {rlabel:<16s} | N={int(reg.nobs):>5d} | R²={reg.rsquared:.4f}')
            print(f'    β(ΔDD_base)={b_base:>8.4f}   β(ΔDD_jump)={b_jump:>8.4f} '
                  f'[t={t_jump:.2f}, p={p_jump:.4f}] {sig}')
        except Exception as e:
            print(f'\n  {label:<18s} | {rlabel:<16s} | ERROR: {e}')

# ---
# ## Approach 3: Residual-Based Test
#
# **Idea:** First strip out all standard risk factors from CDS changes.
# Then test whether the structural model's "jump signal" explains
# the remaining variation — the part that standard factors miss.
#
# 1. **Stage 1:** $\Delta CDS_{it} = \alpha_i + \gamma' X_{it} + \varepsilon_{it}$
# 2. **Stage 2:** $\hat\varepsilon_{it} = \delta_0 + \delta_1 \cdot JumpSignal_{it}
#    + \delta_2 \cdot JumpSignal_{it} \times OilExporter_i
#    + \delta_3 \cdot JumpSignal_{it} \times HighOVX_t
#    + \delta_4 \cdot JumpSignal_{it} \times OilExporter_i \times HighOVX_t + u_{it}$
#
# If $\delta_4 > 0$: the jump component captures sovereign-specific oil risk
# that standard global factors cannot explain.

In [ ]:
# ================================================================
# APPROACH 3: RESIDUAL-BASED TEST
# ================================================================

print('\n' + '=' * 80)
print('  APPROACH 3: RESIDUAL-BASED TEST')
print('  Step 1: Strip standard risk factors from ΔCDS')
print('  Step 2: Test if the jump signal explains the residual')
print('=' * 80)

# ---- Stage 1: Factor model for CDS changes ----
# We build a standard factor model. If you have specific control columns,
# adjust the list below. The code auto-detects what's available.

# Also construct some controls from what we know exists in the panel
if 'OVX' in df_clean.columns:
    df_clean['d_ovx'] = df_clean.groupby('country_clean')['OVX'].diff()
if 'msci_vol_annual' in df_clean.columns:
    df_clean['d_msci_vol'] = df_clean.groupby('country_clean')['msci_vol_annual'].diff()

# Re-check controls including the ones we just created
all_possible = available_controls + ['d_ovx', 'd_msci_vol']
stage1_controls = [c for c in all_possible if c in df_clean.columns
                   and df_clean[c].notna().sum() > len(df_clean) * 0.5]

print(f'\nStage 1 controls: {stage1_controls}')

if not stage1_controls:
    print('\n  WARNING: No control variables found in the panel!')
    print('  The residual-based test works best with controls like:')
    print('  VIX changes, US Treasury changes, MSCI returns, FX returns, etc.')
    print('  Proceeding with country FE only (results will be weaker).\n')

# Build Stage 1 formula
if stage1_controls:
    ctrl_s1 = ' + '.join(stage1_controls)
    fml_s1 = f'd_cds_spread ~ {ctrl_s1} + C(country_clean)'
else:
    fml_s1 = 'd_cds_spread ~ C(country_clean)'

# Drop NAs for the stage 1 regression
s1_cols = ['d_cds_spread', 'country_clean', 'oil_exporter', 'high_ovx',
           'd_dd_jump_increment', 'dd_jump_increment'] + stage1_controls
df_s1 = df_clean.dropna(subset=[c for c in s1_cols if c in df_clean.columns]).copy()

print(f'Stage 1 sample: {len(df_s1)} obs')

reg_s1 = smf.ols(fml_s1, data=df_s1).fit(
    cov_type='cluster', cov_kwds={'groups': df_s1['country_clean']})

print(f'Stage 1 R² = {reg_s1.rsquared:.4f}  (variance explained by standard factors)')

# Print factor coefficients
print(f'\n  {"Factor":<25s} {"Coef":>8s} {"t":>7s} {"p":>7s}')
print('  ' + '-' * 52)
for var in stage1_controls:
    if var in reg_s1.params.index:
        print(f'  {var:<25s} {reg_s1.params[var]:>8.4f} '
              f'{reg_s1.tvalues[var]:>7.2f} {reg_s1.pvalues[var]:>7.4f}')

# Extract residuals
df_s1['resid_cds'] = reg_s1.resid

print(f'\n  Residual std: {df_s1["resid_cds"].std():.2f} bps')
print(f'  This is the CDS variation NOT explained by standard factors.')

In [ ]:
# ---- Stage 2: Does the jump signal explain the residual? ----
print('\n' + '=' * 80)
print('  STAGE 2: Jump Signal → CDS Residual')
print('=' * 80)

# The jump signal: how much the jump model moves DD differently from baseline
# We use the CHANGE in the jump increment (week-to-week)
# Positive d_dd_jump_increment = jumps pushed DD up relative to baseline this week
# Negative = jumps pushed DD down (more distress) relative to baseline

print(f'\nJump signal (Δ[DD_M3 - DD_M1]) summary:')
print(df_s1['d_dd_jump_increment'].describe().to_string())

# --- 3A. Simple: does the jump signal explain the residual? ---
print('\n--- 3A. Unconditional: Jump signal → Residual ---')

fml_s2a = 'resid_cds ~ d_dd_jump_increment'
reg_s2a = smf.ols(fml_s2a, data=df_s1).fit(
    cov_type='cluster', cov_kwds={'groups': df_s1['country_clean']})

b = reg_s2a.params['d_dd_jump_increment']
t = reg_s2a.tvalues['d_dd_jump_increment']
p = reg_s2a.pvalues['d_dd_jump_increment']
sig = '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.1 else ''
print(f'  β(jump_signal) = {b:.4f}  [t = {t:.2f}, p = {p:.4f}] {sig}')
print(f'  R² = {reg_s2a.rsquared:.4f}')

# --- 3B. Full DiD: Jump signal × Oil exporter × High OVX ---
print('\n--- 3B. DiD: Jump Signal × Oil Exporter × High OVX → Residual ---\n')

fml_s2b = ('resid_cds ~ d_dd_jump_increment '
           '+ d_dd_jump_increment : oil_exporter '
           '+ d_dd_jump_increment : high_ovx '
           '+ d_dd_jump_increment : oil_exporter : high_ovx '
           '+ oil_exporter + high_ovx + oil_exporter : high_ovx')

reg_s2b = smf.ols(fml_s2b, data=df_s1).fit(
    cov_type='cluster', cov_kwds={'groups': df_s1['country_clean']})

print(f'  R² = {reg_s2b.rsquared:.4f},  N = {int(reg_s2b.nobs)}\n')

key_vars_s2 = [
    'd_dd_jump_increment',
    'd_dd_jump_increment:oil_exporter',
    'd_dd_jump_increment:high_ovx',
    'd_dd_jump_increment:oil_exporter:high_ovx',
    'oil_exporter',
    'high_ovx',
    'oil_exporter:high_ovx',
]

print(f'  {"Variable":<50s} {"Coef":>8s} {"t":>7s} {"p":>7s}  {"Sig":>3s}')
print('  ' + '-' * 80)
for var in key_vars_s2:
    if var in reg_s2b.params.index:
        coef = reg_s2b.params[var]
        tval = reg_s2b.tvalues[var]
        pval = reg_s2b.pvalues[var]
        sig = '***' if pval < 0.01 else '**' if pval < 0.05 else '*' if pval < 0.1 else ''
        print(f'  {var:<50s} {coef:>8.4f} {tval:>7.2f} {pval:>7.4f}  {sig:>3s}')

print('\n  INTERPRETATION:')
print('  β(jump_signal)                     = baseline effect (controls, low OVX)')
print('  β(... × oil_exporter)              = extra for oil exporters')
print('  β(... × high_ovx)                  = extra during stress')
print('  β(... × oil_exporter × high_ovx)   = DiD: THESIS HYPOTHESIS')
print('  → If significant: oil-calibrated jumps capture sovereign-specific')
print('    oil risk that standard risk factors miss.')

In [ ]:
# --- 3C. Split-sample for robustness ---
print('\n' + '=' * 80)
print('--- 3C. Split-Sample: Jump Signal → Residual ---')
print('=' * 80)

for grp, label in [(1, 'Oil Exporters'), (0, 'Controls')]:
    sub = df_s1[df_s1['oil_exporter'] == grp]
    for regime, rlabel in [('all', 'Full Sample'),
                            ('high', 'High OVX'),
                            ('low', 'Low OVX')]:
        if regime == 'high':
            ss = sub[sub['high_ovx'] == 1]
        elif regime == 'low':
            ss = sub[sub['high_ovx'] == 0]
        else:
            ss = sub

        if len(ss) < 30:
            continue

        reg = smf.ols('resid_cds ~ d_dd_jump_increment', data=ss).fit(
            cov_type='cluster', cov_kwds={'groups': ss['country_clean']})

        b = reg.params.get('d_dd_jump_increment', np.nan)
        t = reg.tvalues.get('d_dd_jump_increment', np.nan)
        p = reg.pvalues.get('d_dd_jump_increment', np.nan)
        sig = '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.1 else ''
        print(f'  {label:<18s} | {rlabel:<12s} | N={int(reg.nobs):>5d} '
              f'| R²={reg.rsquared:.4f} | β={b:>8.4f} [t={t:.2f}] {sig}')

# ---
# ## Summary Table: Collecting Results

In [ ]:
# ================================================================
# COMBINED RESULTS TABLE
# ================================================================
print('\n' + '=' * 80)
print('  COMBINED RESULTS: Does the Jump Component Add Information?')
print('=' * 80)

print(f'\n  {"Test":<55s} {"β":>7s} {"t":>6s} {"p":>7s} {"Sig":>3s}')
print('  ' + '-' * 82)

rows = []

# Approach 1A: nested test
b = reg_nested.params.get('d_dd_jump_increment', np.nan)
t = reg_nested.tvalues.get('d_dd_jump_increment', np.nan)
p = reg_nested.pvalues.get('d_dd_jump_increment', np.nan)
sig = '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.1 else ''
rows.append(('A1. Jump incr. (full panel, nested)', b, t, p, sig))

# Approach 1B: DiD
for var, nice in [
    ('d_dd_jump_increment', 'A1B. Jump incr. (baseline)'),
    ('d_dd_jump_increment:oil_exporter', 'A1B. ... × Oil Exporter'),
    ('d_dd_jump_increment:high_ovx', 'A1B. ... × High OVX'),
    ('d_dd_jump_increment:oil_exporter:high_ovx', 'A1B. ... × Oil × High OVX (DiD)')
]:
    if var in reg_did.params.index:
        rows.append((nice,
                      reg_did.params[var],
                      reg_did.tvalues[var],
                      reg_did.pvalues[var],
                      '***' if reg_did.pvalues[var] < 0.01
                      else '**' if reg_did.pvalues[var] < 0.05
                      else '*' if reg_did.pvalues[var] < 0.1 else ''))

# Approach 3A: residual, unconditional
b = reg_s2a.params['d_dd_jump_increment']
t = reg_s2a.tvalues['d_dd_jump_increment']
p = reg_s2a.pvalues['d_dd_jump_increment']
sig = '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.1 else ''
rows.append(('A3. Jump signal → residual (unconditional)', b, t, p, sig))

# Approach 3B: DiD on residual
for var, nice in [
    ('d_dd_jump_increment', 'A3B. Jump signal → residual (baseline)'),
    ('d_dd_jump_increment:oil_exporter', 'A3B. ... × Oil Exporter'),
    ('d_dd_jump_increment:high_ovx', 'A3B. ... × High OVX'),
    ('d_dd_jump_increment:oil_exporter:high_ovx', 'A3B. ... × Oil × High OVX (DiD)')
]:
    if var in reg_s2b.params.index:
        rows.append((nice,
                      reg_s2b.params[var],
                      reg_s2b.tvalues[var],
                      reg_s2b.pvalues[var],
                      '***' if reg_s2b.pvalues[var] < 0.01
                      else '**' if reg_s2b.pvalues[var] < 0.05
                      else '*' if reg_s2b.pvalues[var] < 0.1 else ''))

for name, b, t, p, sig in rows:
    print(f'  {name:<55s} {b:>7.4f} {t:>6.2f} {p:>7.4f} {sig:>3s}')

print('\n  NOTE: DD is inversely related to CDS spreads.')
print('  A negative β on ΔDD_jump_incr means that when jumps push DD down')
print('  (more distress), CDS spreads increase — the expected sign.')

# ---
# ## Visualization

In [ ]:
# ================================================================
# FIGURE 1: Jump increment effect by group and regime
# ================================================================

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel A: Scatter of jump signal vs CDS residual
ax = axes[0]
for grp, label, color, marker in [(1, 'Oil Exp.', 'firebrick', 'D'),
                                    (0, 'Control', 'steelblue', 'o')]:
    sub = df_s1[df_s1['oil_exporter'] == grp].sample(
        min(500, len(df_s1[df_s1['oil_exporter'] == grp])), random_state=42)
    ax.scatter(sub['d_dd_jump_increment'], sub['resid_cds'],
               c=color, marker=marker, s=10, alpha=0.3, label=label)

ax.axhline(0, color='gray', lw=0.5, ls=':')
ax.axvline(0, color='gray', lw=0.5, ls=':')
ax.set_xlabel('Δ(DD_M3 - DD_M1)')
ax.set_ylabel('CDS residual (bps)')
ax.set_title('A. Jump Signal vs CDS Residual')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.2)

# Panel B: Bar chart of β by group × regime
ax = axes[1]
betas = {}
for grp, label in [(1, 'Oil Exp.'), (0, 'Control')]:
    for regime, rlabel in [(1, 'High OVX'), (0, 'Low OVX')]:
        sub = df_s1[(df_s1['oil_exporter'] == grp) & (df_s1['high_ovx'] == regime)]
        if len(sub) < 30:
            betas[(label, rlabel)] = (np.nan, np.nan)
            continue
        try:
            reg = smf.ols('resid_cds ~ d_dd_jump_increment', data=sub).fit(
                cov_type='cluster', cov_kwds={'groups': sub['country_clean']})
            betas[(label, rlabel)] = (reg.params['d_dd_jump_increment'],
                                       reg.bse['d_dd_jump_increment'] * 1.96)
        except:
            betas[(label, rlabel)] = (np.nan, np.nan)

labels_b = ['Oil Exp.\nHigh OVX', 'Oil Exp.\nLow OVX',
            'Control\nHigh OVX', 'Control\nLow OVX']
vals = [betas[('Oil Exp.', 'High OVX')],
        betas[('Oil Exp.', 'Low OVX')],
        betas[('Control', 'High OVX')],
        betas[('Control', 'Low OVX')]]
colors = ['firebrick', 'lightsalmon', 'steelblue', 'lightblue']

x_pos = range(len(labels_b))
for i, (v, ci) in enumerate(vals):
    if not np.isnan(v):
        ax.bar(i, v, color=colors[i], edgecolor='k', linewidth=0.5)
        ax.errorbar(i, v, yerr=ci, fmt='none', color='k', capsize=4)

ax.set_xticks(x_pos)
ax.set_xticklabels(labels_b, fontsize=8)
ax.axhline(0, color='gray', lw=0.8, ls='-')
ax.set_ylabel('β (Jump Signal → CDS Residual)')
ax.set_title('B. Effect by Group × Regime')
ax.grid(True, alpha=0.2, axis='y')

# Panel C: Time series of jump increment for one oil exporter
ax = axes[2]
# Pick the first available oil exporter
oil_countries = sorted(df_s1[df_s1['oil_exporter']==1]['country_clean'].unique())
if oil_countries:
    show_ctry = oil_countries[0]  # e.g., 'Brazil' or 'Colombia'
    dc = df_s1[df_s1['country_clean'] == show_ctry].sort_values('date')

    ax2 = ax.twinx()
    ax.plot(dc['date'], dc['dd_jump_increment'], color='tomato',
            lw=1.2, label='DD jump increment')
    ax2.plot(dc['date'], dc['OVX'], color='orange', lw=0.8, alpha=0.5,
             label='OVX')

    # Shade high OVX periods
    high_mask = dc['high_ovx'] == 1
    for _, row in dc[high_mask].iterrows():
        ax.axvspan(row['date'] - pd.Timedelta(days=3),
                   row['date'] + pd.Timedelta(days=3),
                   alpha=0.05, color='red')

    ax.set_ylabel('DD(M3) - DD(M1)', color='tomato')
    ax2.set_ylabel('OVX', color='orange')
    ax.set_title(f'C. {show_ctry} — Jump Increment & OVX')
    ax.axhline(0, color='gray', lw=0.5, ls=':')
    ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig('output/fig_structural_test.png', dpi=150, bbox_inches='tight')
plt.show()
print('  Saved: output/fig_structural_test.png')

# ---
# ## Regression Tables for Thesis
#
# Clean LaTeX-style output for the results section.

In [ ]:
# ================================================================
# FORMATTED TABLE FOR THESIS (can paste into LaTeX)
# ================================================================

print('\n% ---- LaTeX Table: Approach 1 (Nested) ----')
print(r'\begin{table}[htbp]')
print(r'\centering')
print(r'\caption{Nested Change-on-Change Regression: Does the Jump Component Add Information?}')
print(r'\label{tab:nested_coc}')
print(r'\begin{tabular}{lccc}')
print(r'\toprule')
print(r' & (1) Baseline & (2) + Jump Incr. & (3) DiD \\')
print(r'\midrule')

# ΔDD baseline
for label, reg_obj in [('(1)', reg_base), ('(2)', reg_nested), ('(3)', reg_did)]:
    pass  # we'll build row by row

def fmt_coef(reg, var):
    if var not in reg.params.index:
        return ''
    b = reg.params[var]
    se = reg.bse[var]
    p = reg.pvalues[var]
    stars = '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.1 else ''
    return f'{b:.4f}{stars} ({se:.4f})'

var_rows = [
    (r'$\Delta DD^{baseline}$', 'd_dd_m1'),
    (r'$\Delta DD^{jump}$', 'd_dd_jump_increment'),
    (r'$\Delta DD^{jump} \times OilExporter$', 'd_dd_jump_increment:oil_exporter'),
    (r'$\Delta DD^{jump} \times HighOVX$', 'd_dd_jump_increment:high_ovx'),
    (r'$\Delta DD^{jump} \times Oil \times HighOVX$', 'd_dd_jump_increment:oil_exporter:high_ovx'),
]

for nice, var in var_rows:
    c1 = fmt_coef(reg_base, var)
    c2 = fmt_coef(reg_nested, var)
    c3 = fmt_coef(reg_did, var)
    if c1 or c2 or c3:
        print(f'{nice} & {c1} & {c2} & {c3} \\\\')

print(r'\midrule')
print(f'Country FE & Yes & Yes & Yes \\\\')
ctrl_yn = 'Yes' if available_controls else 'No'
print(f'Controls & {ctrl_yn} & {ctrl_yn} & {ctrl_yn} \\\\')
print(f'R$^2$ & {reg_base.rsquared:.4f} & {reg_nested.rsquared:.4f} & {reg_did.rsquared:.4f} \\\\')
print(f'N & {int(reg_base.nobs)} & {int(reg_nested.nobs)} & {int(reg_did.nobs)} \\\\')
print(r'\bottomrule')
print(r'\end{tabular}')
print(r'\end{table}')

print('\n% ---- LaTeX Table: Approach 3 (Residual-Based) ----')
print(r'\begin{table}[htbp]')
print(r'\centering')
print(r'\caption{Residual-Based Test: Jump Signal and Unexplained CDS Variation}')
print(r'\label{tab:residual_test}')
print(r'\begin{tabular}{lcc}')
print(r'\toprule')
print(r' & (1) Unconditional & (2) DiD \\')
print(r'\midrule')

var_rows_s2 = [
    (r'$JumpSignal$', 'd_dd_jump_increment'),
    (r'$JumpSignal \times OilExporter$', 'd_dd_jump_increment:oil_exporter'),
    (r'$JumpSignal \times HighOVX$', 'd_dd_jump_increment:high_ovx'),
    (r'$JumpSignal \times Oil \times HighOVX$', 'd_dd_jump_increment:oil_exporter:high_ovx'),
    (r'$OilExporter$', 'oil_exporter'),
    (r'$HighOVX$', 'high_ovx'),
]

for nice, var in var_rows_s2:
    c1 = fmt_coef(reg_s2a, var)
    c2 = fmt_coef(reg_s2b, var)
    if c1 or c2:
        print(f'{nice} & {c1} & {c2} \\\\')

print(r'\midrule')
print(f'Stage 1 R$^2$ & \\multicolumn{{2}}{{c}}{{{reg_s1.rsquared:.4f}}} \\\\')
print(f'Stage 2 R$^2$ & {reg_s2a.rsquared:.4f} & {reg_s2b.rsquared:.4f} \\\\')
print(f'N & {int(reg_s2a.nobs)} & {int(reg_s2b.nobs)} \\\\')
print(r'\bottomrule')
print(r'\end{tabular}')
print(r'\end{table}')

print('\n  Done. Copy the LaTeX above into your thesis.')